<a href="https://colab.research.google.com/github/1Bur1/Tuwaiq-classes---advanced-AI-python---final-project/blob/main/01_cleaning%20check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1 — Load, Explore & Clean
**Dataset:** Ames Housing Dataset  
**Goal:** Load the data, understand what each column means, and clean it up so we end up with a clean DataFrame ready for analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---
## Task 1 — Load the Data
We use `pd.read_csv()` to load the dataset and print the first 5 rows to get a quick look at what we're working with.

In [3]:
import pandas as pd

df = pd.read_csv("AmesHousing.csv")
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


---
## Task 2 — Check the Shape
`.shape` tells us how many rows and columns are in the dataset.

In [5]:
df.shape

(2930, 82)

---
## Task 3 — Inspect & Fix Data Types
We use `.info()` to check all column types. Some columns are stored as the wrong type:

- **`MS SubClass`** is stored as `int64` but it's actually a category code
- **`Bsmt Full Bath`, `Bsmt Half Bath`, `Garage Cars`** are stored as `float64` but they represent whole-number counts. They should be `Int64` (which allows missing values).
- **`Year Built`, `Year Remod/Add`, `Yr Sold`** represent years and are better stored as `datetime` so we can do time-based math later.

In [ ]:
df.info()

In [ ]:
# MS SubClass is a category code, not a real number
df['MS SubClass'] = df['MS SubClass'].astype('category')



#Bath and garage counts should be integers (Int64 supports missing values)
for col in ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')



# Year columns should be datetime
for col in ['Year Built', 'Year Remod/Add', 'Yr Sold']:
    df[col] = pd.to_datetime(df[col], format='%Y', errors='coerce')



#All text (object) columns → category to save memory
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')




df.info()

---
## Task 4 — Find & Handle Missing Values
We check which columns have the most missing values. Then we decide what to do:

| Column | Missing | Decision | Reason |
|--------|---------|----------|---------|
| Pool QC | ~99% | **Drop** | Almost entirely missing — not useful |
| Misc Feature | ~96% | **Drop** | Same reason |
| Alley | ~93% | **Drop** | Most houses have no alley — not informative |
| Fence | ~80% | **Drop** | Too many missing to reliably fill |
| Categorical columns | small % | **Fill with mode** | Use the most common value |
| Numeric columns | small % | **Fill with median** | Median is better than mean when data is skewed |

In [ ]:
# Show the top 10 columns with the most missing values
missing = df.isnull().sum().sort_values(ascending=False)
print("top 10 columns with most missing values:")
print(missing[missing > 0].head(10))

In [ ]:
# Drop columns with too many missing values (more than 50% missing)
cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Dropped columns: {cols_to_drop}")




# Fill categorical columns with the mode (most frequent value)
for col in df.select_dtypes(include='category').columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])




# Fill numeric columns with the median (robust to outliers)
for col in df.select_dtypes(include=['int64', 'float64', 'Int64']).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())




# Fill datetime columns with the mode
for col in df.select_dtypes(include='datetime64[ns]').columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])




print(f"Missing values remaining: {df.isnull().sum().sum()}")

---
## Task 5 — Handle Duplicates
We check if any rows are exact duplicates and remove them if found.

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of duplicated rows: {num_duplicates}")

df = df.drop_duplicates()
print(f"Rows after removing duplicates: {df.shape[0]}")

---
## Task 6 — Spot & Handle Outliers
We look at the target column `SalePrice` using a **boxplot**. The boxplot shows us dots above the whisker — those are outliers (very expensive houses).

**Decision:** We cap values at the 99th percentile. This means any house priced above that limit gets clipped to the limit. This prevents extremely rare expensive houses from messing up our future model.

In [ ]:
# Boxplot BEFORE capping
plt.figure(figsize=(6, 4))
plt.boxplot(df['SalePrice'])
plt.title('SalePrice Boxplot — BEFORE Capping')
plt.ylabel('Price ($)')
plt.show()

print(f"Max SalePrice before capping: ${df['SalePrice'].max():,}")

In [ ]:
# Cap SalePrice at the 99th percentile
upper_99 = df['SalePrice'].quantile(0.99)
print(f"99th percentile value: ${upper_99:,}")

df['SalePrice'] = df['SalePrice'].clip(upper=upper_99)

print(f"Max SalePrice AFTER capping: ${df['SalePrice'].max():,}")

# Boxplot AFTER capping — to confirm it worked
plt.figure(figsize=(6, 4))
plt.boxplot(df['SalePrice'])
plt.title('SalePrice Boxplot — AFTER Capping at 99th Percentile')
plt.ylabel('Price ($)')
plt.show()

---
## Task 7 — The `clean_data()` Function
We wrap all the steps above into a single reusable function. This way we can clean any new dataset with one line of code.

In [6]:
import pandas as pd

def clean_data(filepath):
    """
    Loads and cleans the Ames Housing dataset.
    Steps: fix data types, handle missing values,
    remove duplicates, and cap outliers.
    Returns a clean DataFrame.
    """
    # --- Step 1 & 2: Load & check shape ---
    df = pd.read_csv(filepath)
    print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")

    # --- Step 3: Fix data types ---
    df['MS SubClass'] = df['MS SubClass'].astype('category')

    for col in ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Cars']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    for col in ['Year Built', 'Year Remod/Add', 'Yr Sold']:
        df[col] = pd.to_datetime(df[col], format='%Y', errors='coerce')

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')





    # --- Step 4: Handle missing values ---
    # Drop columns with more than 50% missing
    cols_to_drop = ['Pool QC', 'Misc Feature', 'Alley', 'Fence']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])


    # Fill categorical with mode
    for col in df.select_dtypes(include='category').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])


    # Fill numeric with median
    for col in df.select_dtypes(include=['int64', 'float64', 'Int64']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())


    # Fill datetime with mode
    for col in df.select_dtypes(include='datetime64[ns]').columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])




    # --- Step 5: Remove duplicates ---
    before = df.shape[0]
    df = df.drop_duplicates()
    print(f"Removed {before - df.shape[0]} duplicate rows")




    # --- Step 6: Cap outliers at 99th percentile ---
    upper_99 = df['SalePrice'].quantile(0.99)
    df['SalePrice'] = df['SalePrice'].clip(upper=upper_99)

    print("Cleaning complete!")
    return df


# Run the function
df_clean = clean_data("AmesHousing.csv")
df_clean.head()

Loaded data: 2930 rows, 82 columns
Removed 0 duplicate rows
Cleaning complete!


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,5,2010-01-01,WD,Normal,215000.0
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,0,120,0,0,6,2010-01-01,WD,Normal,105000.0
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,0,12500,6,2010-01-01,WD,Normal,172000.0
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,0,4,2010-01-01,WD,Normal,244000.0
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,3,2010-01-01,WD,Normal,189900.0


---
## Task 8 — Final Checks
Before moving to the next phase, we run 3 checks to make sure our data is clean and ready.

In [8]:
#No missing values in the target column (SalePrice)
assert df_clean['SalePrice'].isnull().sum() == 0

# Check 2: All sale prices are positive (a price of 0 or negative makes no sense)
assert (df_clean['SalePrice'] > 0).all()

# Check 3: We have the correct number of columns after dropping 4
expected_cols = 82 - 4 + 0  # original 82 minus 4 dropped (Pool QC, Misc Feature, Alley, Fence)
assert df_clean.shape[1] == expected_cols, f"Expected {expected_cols} columns, got {df_clean.shape[1]}!"

